In [1]:
import json

import gc_utils
import h5py
import halo_analysis as halo
import matplotlib.pyplot as plt
import numpy as np
import utilities as ut
from scipy.optimize import minimize_scalar

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

# SMALL_SIZE = 10
# MEDIUM_SIZE = 10
# BIGGER_SIZE = 10

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [2]:
# sim = "m12i"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"
sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [81]:
sim = "m12b"

fir_dir = sim_dir + sim + "/" + sim + "_res7100"

halt = gc_utils.get_halo_tree(sim, sim_dir)
hal = halo.io.IO.read_catalogs("redshift", 0, fir_dir)
part = gc_utils.open_snapshot(600, fir_dir, ["star", "gas", "dark"])

hidx = np.where(halt["tid"] == sim_data[sim]["halo"])[0][0]

Retrieving Snapshot 600..................:   0%|                                                                                | 0/1 [00:08<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
msk_20kpc = part["star"].prop("host.principal.distance.total") < 20
np.round(np.sum(part["star"]["mass"][msk_20kpc]) * 1e-10 * 0.9, 1)

6.3

In [ ]:
# stars within 20 kpc
radi = part["star"].prop("host.principal.distance.total")[msk_20kpc].astype(np.float64)
mass = part["star"]["mass"][msk_20kpc].astype(np.float64)

# sort by radius
idx = np.argsort(radi)
r_sorted = radi[idx]
m_sorted = mass[idx]

# cumulative enclosed mass
cum_mass = np.cumsum(m_sorted)

# 90% of the mass inside 20 kpc
target = 0.9 * cum_mass[-1]
# target = 0.9 * np.sum(part["star"]["mass"][msk_20kpc])

# interpolate to get r90
r90 = np.interp(target, cum_mass, r_sorted)
np.round(cum_mass[-1] * 1e-10, 2), np.round(0.9 * cum_mass[-1] * 1e-10, 2), np.round(r90, 2)

(7.0, 6.3, 9.98)

In [ ]:
halt["mass.vir"][hidx] * 1e-12

0.903703691264

In [ ]:
hal["mass.vir"][hal["host.index"][0]] * 1e-12

0.903703691264

# total virial mass

In [ ]:
def find_rvir(r, m, rho_vir):

    def objective(r0):
        m_enc = np.sum(m[r <= r0])
        rho_enc = m_enc / ((4 / 3) * np.pi * r0**3)
        return abs(rho_enc - rho_vir)

    result = minimize_scalar(
        objective,
        bounds=(1.0, np.max(r)),
        method="bounded",
    )

    return result.x

In [ ]:
part = gc_utils.open_snapshot(600, fir_dir, ["star", "gas", "dark"])

In [ ]:
omega_m = part.info["omega_matter"]
omega_l = part.info["omega_lambda"]
z = part.info["redshift"]
h = part.info["hubble"]

E2 = omega_m * (1 + z) ** 3 + omega_l
omega_m_z = omega_m * (1 + z) ** 3 / E2

x = omega_m_z - 1.0

delta_vir = 18.0 * np.pi**2 + 82.0 * x - 39.0 * x**2

rho_vir = delta_vir * ut.constant.density_critical_0 * h**2

In [ ]:
rd = part["dark"].prop("host.principal.distance.total").astype(np.float64)
md = part["dark"]["mass"].astype(np.float64)

rs = part["star"].prop("host.principal.distance.total").astype(np.float64)
ms = part["star"]["mass"].astype(np.float64)

rg = part["gas"].prop("host.principal.distance.total").astype(np.float64)
mg = part["gas"]["mass"].astype(np.float64)

r = np.concatenate((rd, rs, rg))
m = np.concatenate((md, ms, mg))

In [ ]:
rvir = find_rvir(r, m, rho_vir)

In [ ]:
mvir = np.sum(m[r <= rvir])

In [ ]:
rvir, mvir * 1e-12

(268.0443000421347, 1.07312055712126)

In [ ]:
halt["mass.vir"][hidx] * 1e-12

0.903703691264

In [ ]:
np.sum(md[rd <= rvir]) * 1e-12

0.93659397816875

In [ ]:
np.abs(halt["mass.vir"][hidx] - np.sum(md[rd <= rvir])) / halt["mass.vir"][0]

0.036394990108701154

CHECK REPLACEMENT MASS

In [33]:
sim = "m12b"
fir_dir = sim_dir + sim + "/" + sim + "_res7100"
part = gc_utils.open_snapshot(600, fir_dir, ["star"])
smass = part["star"]["mass"].copy()
sr = part["star"].prop("host.distance.principal.total").copy()

ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
ghost_data = h5py.File(ghost_file, "r")

lst = []
for it_id in ghost_data.keys():
    src_dat = ghost_data[it_id]["source"]
    pidxs_600 = src_dat["pidx_snap600"][()]
    ptypes = np.array([ptype.decode("utf-8") for ptype in src_dat["ptype"][()]])
    gc_smsk = ptypes == "star"

    log_mfor = src_dat["logm_tfor"][()]
    mfor = 10**log_mfor
    mfor_ev = 0.55 * mfor
    mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
    mdifs = mfor_ev - mz0

    gc_r = np.linalg.norm(src_dat["pxyz_snap600"][()], axis=1)

    # stellar mass in/from GCs
    smass[pidxs_600[gc_smsk]] = mfor_ev[gc_smsk]

    smass_20kpc = smass[sr < 20].astype(np.float64)
    smass_20kpc = np.append(smass_20kpc, mfor_ev[~gc_smsk & (gc_r < 20)])
    lst.append(np.sum(smass_20kpc) * 1e-10)

np.mean(lst)

Retrieving Snapshot 600..................: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:13<00:00, 13.81s/it]


9.637678991235262

In [35]:
100 * (4.42 * 10**8) / (9.42 * 10**10)

0.4692144373673036

In [36]:
100 * (4.42 * 10**8) / (9.64 * 10**10)

0.45850622406639

In [41]:
100 * (4.57 * 10**8) / (13.3 * 10**10)

0.343609022556391

In [105]:
100 * (9.6 * 10**7) / (5 * 10**10)

0.192

In [31]:
sim = "m12m"
fir_dir = sim_dir + sim + "/" + sim + "_res7100"
part = gc_utils.open_snapshot(600, fir_dir, ["star"])
smass = part["star"]["mass"].copy()
sr = part["star"].prop("host.distance.principal.total").copy()

np.round(np.sum(smass[sr < 20]) * 1e-10, 2), np.round(np.sum(smass[sr < 296]) * 1e-10, 2)

Retrieving Snapshot 600..................: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:20<00:00, 20.18s/it]


(12.62, 13.31)